In [12]:
import sys
sys.path.append("..")

import os
import json
import numpy as np
import joblib
import wfdb

from src import portable
from src.features import build_features
from src.pipeline import analyze_portable

bundle = joblib.load("../models/ecg_rf_small.joblib")
model = bundle["model"]
sos_main, sos_det = portable.make_sos()

# ---------- 1) export the model and the filter settings ----------
os.makedirs("../export", exist_ok=True)
cfg = {
    "fs": portable.FS, "pre": portable.PRE, "post": portable.POST,
    "classes": ["N", "S", "V", "F"],
    "decision_weights": [float(v) for v in bundle["decision_weights"]],
    "sos_main": sos_main.tolist(),
    "sos_det": sos_det.tolist(),
}
with open("../export/ecg_forest.json", "w") as f:
    json.dump(portable.export_forest(model), f, separators=(",", ":"))
with open("../export/ecg_config.json", "w") as f:
    json.dump(cfg, f, indent=1)
print("forest json:", round(os.path.getsize("../export/ecg_forest.json") / 1e6, 2), "MB")

# read them back: every check below uses only what is stored in the files
forest = json.load(open("../export/ecg_forest.json"))
cfg = json.load(open("../export/ecg_config.json"))

# ---------- 2) features and forest on 3000 random test beats ----------
d = np.load("../data/processed/beats.npz")
m = d["y_test"] != 4
X, rr, g = d["X_test"][m], d["rr_test"][m], d["g_test"][m]
F = build_features(X, rr, g)
med = {r: float(np.median(rr[g == r][:, 0])) for r in np.unique(g)}

idx = np.random.default_rng(0).choice(len(X), 3000, replace=False)
Fp = np.array([portable.beat_features(X[i], rr[i][0], rr[i][1], med[g[i]]) for i in idx])
rel = np.abs(F[idx] - Fp) / (np.abs(F[idx]) + 1e-9)
print("features: max abs diff =", np.abs(F[idx] - Fp).max(), "| max relative diff =", rel.max())

sk = model.predict_proba(F[idx])
mine = np.array([portable.forest_predict_proba(forest, f) for f in Fp])
w = np.array(cfg["decision_weights"])
print("forest: max abs diff =", np.abs(sk - mine).max())
print("same predicted class:", (np.argmax(sk * w, 1) == np.argmax(mine * w, 1)).mean())

# ---------- 3) full pipeline on raw signals ----------
for rec in ["100", "200", "233"]:
    record = wfdb.rdrecord(f"../data/mitdb/{rec}")
    sig = record.p_signal[:, record.sig_name.index("MLII")]
    a = analyze_portable(sig, record.fs, bundle)               # sklearn version
    b = portable.analyze_reference(sig, record.fs, cfg, forest)  # simple version
    same_peaks = np.array_equal(a["peaks"], b["peaks"])
    same_pred = same_peaks and np.array_equal(a["pred"], b["pred"])
    print(rec, "| beats:", len(a["peaks"]),
          "| same peaks:", same_peaks, "| same predictions:", same_pred)

forest json: 1.28 MB
features: max abs diff = 1.0658141036401503e-14 | max relative diff = 1.0805595304381129e-13
forest: max abs diff = 1.0586653143018765e-07
same predicted class: 1.0
100 | beats: 2271 | same peaks: True | same predictions: True
200 | beats: 2598 | same peaks: True | same predictions: True
233 | beats: 3076 | same peaks: True | same predictions: True


In [13]:
# ---------- 4) golden test vectors for the phone app ----------
# (run after the checks above: this cell stops with an error if the two paths disagree)
cases = {}
for rec, start_s, dur_s in [("200", 100, 30), ("100", 60, 20), ("233", 300, 30)]:
    record = wfdb.rdrecord(f"../data/mitdb/{rec}")
    sig = record.p_signal[:, record.sig_name.index("MLII")]
    seg = np.round(sig[start_s * 360:(start_s + dur_s) * 360], 4)   # input = rounded values

    a = analyze_portable(seg, 360, bundle)                          # sklearn version
    b = portable.analyze_reference(seg, 360, cfg, forest)           # simple version
    assert np.array_equal(a["peaks"], b["peaks"]) and np.array_equal(a["pred"], b["pred"]), \
        f"record {rec}: the two paths disagree"

    sig_f = b["signal"]
    all_peaks = portable.detect_rpeaks(sig_f, np.array(cfg["sos_det"]))
    pos = {int(p): k for k, p in enumerate(all_peaks)}
    prev_list = [int(all_peaks[pos[int(p)]] - all_peaks[pos[int(p)] - 1]) for p in b["peaks"]]
    med = float(np.median(prev_list))

    beats = []
    for p in b["peaks"][:12]:
        k = pos[int(p)]
        beat = np.round(sig_f[p - 90:p + 110], 9)
        prev_rr = int(all_peaks[k] - all_peaks[k - 1])
        next_rr = int(all_peaks[k + 1] - all_peaks[k])
        feats = portable.beat_features(beat, prev_rr, next_rr, med)
        beats.append({
            "beat": beat.tolist(), "prev_rr": prev_rr, "next_rr": next_rr,
            "median_rr": med, "features": feats.tolist(),
            "proba": np.round(portable.forest_predict_proba(forest, feats), 7).tolist(),
        })

    cases[f"record_{rec}_{start_s}s"] = {
        "raw": seg.tolist(),
        "filtered": np.round(sig_f, 9).tolist(),
        "detected_peaks": [int(p) for p in all_peaks],
        "used_peaks": [int(p) for p in b["peaks"]],
        "pred": [int(v) for v in b["pred"]],
        "proba": np.round(b["proba"], 7).tolist(),
        "beat_tests": beats,
    }
    print(rec, "| samples:", len(seg), "| beats:", len(b["peaks"]),
          "| classes N/S/V/F:", np.bincount(b["pred"], minlength=4).tolist())

out = {
    "notes": {
        "fs": 360,
        "tolerances": {"filtered": 1e-6, "features": "abs(a-b) <= 1e-6 + 1e-6*abs(b)", "proba": 1e-5,
                       "peaks_and_pred": "exact"},
        "classes": ["N", "S", "V", "F"],
        "decision_weights": cfg["decision_weights"],
    },
    "cases": cases,
}
with open("../export/test_vectors.json", "w") as f:
    json.dump(out, f, separators=(",", ":"))
print("test_vectors.json:", round(os.path.getsize("../export/test_vectors.json") / 1e6, 2), "MB")

200 | samples: 10800 | beats: 39 | classes N/S/V/F: [19, 0, 20, 0]
100 | samples: 7200 | beats: 23 | classes N/S/V/F: [23, 0, 0, 0]
233 | samples: 10800 | beats: 49 | classes N/S/V/F: [32, 0, 17, 0]
test_vectors.json: 0.68 MB


In [14]:
from src import pipeline

print("running code reads decision_weights from the model:",
      "decision_weights" in pipeline._classify.__code__.co_consts)
print("peaks:", len(a["peaks"]), len(b["peaks"]), "| same peaks:", np.array_equal(a["peaks"], b["peaks"]))
if len(a["peaks"]) == len(b["peaks"]):
    print("proba max diff:", np.abs(a["proba"] - b["proba"]).max())
    print("different predictions:", int((a["pred"] != b["pred"]).sum()), "of", len(a["pred"]))

running code reads decision_weights from the model: True
peaks: 49 49 | same peaks: True
proba max diff: 4.5260569042637755e-08
different predictions: 0 of 49


In [15]:
import importlib
from src import pipeline

print("module file:", pipeline.__file__)
print("occurrences of 'decision_weights' in that file on disk:",
      open(pipeline.__file__, encoding="utf-8").read().count("decision_weights"))

importlib.reload(pipeline)
from src.pipeline import analyze_portable      # re-bind the name used in the other cells

print("running code now reads decision_weights:",
      "decision_weights" in pipeline._classify.__code__.co_consts)

module file: e:\Projects\ecg-arrhythmia-classification\notebooks\..\src\pipeline.py
occurrences of 'decision_weights' in that file on disk: 1
running code now reads decision_weights: True
